# 04 — Model training

## What this notebook does
Trains **LightGBM**, **XGBoost**, and **CatBoost** with **spatial GroupKFold**. Optionally **RegressorChain** (multi-target) and **stacking** (ensemble).

## Why these models
Gradient boosting handles **nonlinear** and **mixed-scale** tabular features common in environmental stacks (reflectance, climate, flow).

## RegressorChain
Predicts targets **in sequence**; later targets use earlier predictions as inputs. Useful when variables are **chemically coupled** (e.g. salinity and major ions).

## Stacking ensemble — advantages
- **Diversity:** LGB, XGB, Cat optimize different loss landscapes; errors are partially uncorrelated.
- **Robustness:** Meta-learner (e.g. Ridge) weights base models → often better than single best or simple average.
- **Variance reduction:** Similar motivation to bagging, but with learned combination.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
import sys
sys.path.insert(0, str(ROOT))

from src.validation.spatial_cv import SpatialGroupKFold
from src.features.preprocess import feature_columns, impute_median
from src.features.scientific_interactions import add_scientific_interactions
from src.models.multi_target import fit_regressor_chain_multioutput, fit_stacking_per_target

RAW = ROOT / "data" / "raw"
for name in ["water_quality_dataset_v1.csv", "water_quality.csv"]:
    csv_path = RAW / name
    if csv_path.exists():
        break
else:
    raise FileNotFoundError("No data/raw/water_quality_dataset_v1.csv or water_quality.csv found.")
df = pd.read_csv(csv_path)
df = add_scientific_interactions(df)
lat = df["lat"] if "lat" in df.columns else df["Latitude"]
lon = df["lon"] if "lon" in df.columns else df["Longitude"]
targets = [c for c in ["total_alkalinity","electrical_conductance","dissolved_reactive_phosphorus"] if c in df.columns]
cols = feature_columns(df, exclude_targets=True)
X = df[cols].copy()
X, _ = impute_median(X)
Y = df[targets].values
sgkf = SpatialGroupKFold(n_splits=5, n_clusters=30)

def one_target_stacking(j):
    y = Y[:, j]
    preds = np.zeros(len(y))
    for tr, te in sgkf.split(np.zeros((len(y),1)), lat=lat.values, lon=lon.values):
        sc = StandardScaler().fit(X.iloc[tr])
        Xtr, Xte = sc.transform(X.iloc[tr]), sc.transform(X.iloc[te])
        m = fit_stacking_per_target(Xtr, y[tr])
        preds[te] = m.predict(Xte)
    return r2_score(y, preds), np.sqrt(mean_squared_error(y, preds))

for j, name in enumerate(targets):
    r2, rmse = one_target_stacking(j)
    print(name, "spatial CV R2=", round(r2,3), "RMSE=", round(rmse,3))

# RegressorChain on full scaled data (illustrative — use inner CV for production)
sc = StandardScaler().fit(X)
chain = fit_regressor_chain_multioutput(sc.transform(X), Y)
print("RegressorChain train R2 (in-sample):", [round(r2_score(Y[:,k], chain.predict(sc.transform(X))[:,k]),3) for k in range(Y.shape[1])])

c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature nam

total_alkalinity spatial CV R2= 0.335 RMSE= 60.899


c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature nam

electrical_conductance spatial CV R2= 0.268 RMSE= 292.456


c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature nam

dissolved_reactive_phosphorus spatial CV R2= 0.018 RMSE= 50.513


c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\bever\Desktop\ey-Agua\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature nam

RegressorChain train R2 (in-sample): [0.943, 0.935, 0.822]
